<center> <b> Build Architecture

> previous part we talk about custom dataset n dataloader, here we explore about `nn.Module` to undstand how model constructed in torch?

- `nn.Module` -> blueprint for all neur nets component
- all layer model in torch like linear,Conv2d, LayerNorm, TransformerEncoder it child from nn.Module
- we use nn.Module due to superpowers it brings, it allows torch automatically calculate trainable weights and determine which grads need to be calculated?

# Structure (basic)

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__() #<- running constuctor parent class nn.Module

    def forward(self,x):
        return x

## \__init__() & \__forward__()

In [ ]:
# init : layer definition
self.linear = nn.Linear(128,64)
# means: build layer -> set params -> params registration

In [ ]:
# forward: computational flow
x = self.linear(x)
# means: data passing the layer -> actual computation

In [3]:
# exmple again
class SimpleNN(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4,2)

    def forward(self,x):
        x = self.linear(x)
        return x

<style color='red'>vfdsf

# <font color='red'>  IMPORTANT - How to Reverse engineer paper to code?

most of the paper in ai field just combination from some of concept: matmul, linear, normalization, activation, attention etc.

e.g. we see $y = W_2\sigma(W_1x)$ <br>
we can unstand:
- $W_1x$ = Linear
- $\sigma$ = activation func
- $W_2$ = Linear also <br>
so in python we can write
    - Linear
        - Act func
        - Linear

In [ ]:
# code implementation
self.fc1 = nn.Linear(d,hidden)
self.actfunc = nn.GELU()
self.fc2 = nn.Linear(hidden,d)

> for instance the concept of FFN Transformer:

$FFN(x) = max(0,xW_1+b_1)W_2+b_2$

In [4]:
# in torch u can write as simple
class FFN(nn.Module):
    
    def __init__(self,d_model,hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(d_model,hidden_dim)
        self.leaky_relu = nn.LeakyReLU()
        self.fc2 = nn.Linear(hidden_dim,d_model)

    def forward(self,x):
        x = self.fc1(x)
        x = self.leaky_relu(x)
        x = self.fc2(x)

        return x

### Mapping paper notation 🤨
- $Wx+b$ `nn.Linear(in_dim,out_dim)`
- $\sigma(x)$ `nn.ReLu()` or whatever act func
- $LayerNorm(x)$ `nn.LayerNorm(d_model)`
- $x+f(x)$ `x = x+self.block(x)` <- residual connection hmm

In [3]:
# example of resnet block (👆🏻)
class ResNetBlock(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.fc = nn.Linear(dim,dim)

    def forward(self,x):
        out = self.fc(x)
        out = out + x
        return out

# Shape Tracking 101

## LM

let say we've sentences: "i love my mom so much" <br>
tokenizer => [101,1034,3214,2293, 2784, 4083,2022,102] => token_length = 8
- batch = 32
- max seq len = 128
- initial tensor become (B,T) = (32,128) -> tensor contains integer token ids (not yet embding)


Embedding layer<br>
`nn.Embedding(vocab_size,d_model)` -> e.g. `nn.Embedding(30000, 512)`
<br> each token is a 512 embedding dim


Shape after embedding phase<br>
input (32,128) -> out (32,128,512) (batch, token_cnt, embedding_dim)


> <b>Formula <br>

$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

### Attention

- input: (32,128,512)
- Transformers build vector: key, query, value
- QKV Projection
    ```
    self.q_proj = nn.Linear(512,512)
    self.k_proj = nn.Linear(512,512)
    self.v_proj = nn.Linear(512,512)
    ```
- out will same in Linear(512,512) so for each Q,K,V -> (32,128,512)

### Multihead

but.. remember that transformers have multihead architecture so we seperate the embedding dim into each of head
- embedding_dim = 512
- num_heads = 8 <br><br>
head_dim = $\frac{512}{8} = 64$ 
- reshape multihead => initial: (32,128,512) to (32,128,8,64) (B,T,n_head,emb_dim)
- why we need to seperate into many head? so that each head can learn such as..
    - syntax
    - name
    - semantic
    - long dependency
- transpose again (32,128,8,64) (B,T,n_head,emb_dim) -> (32,8,128,64) (B,n_head,T,emb_dim)


### Attention score

Formula: $QK^{T}$
- Q:(32,8,128,64)
- K:(32,8,128,64) -> $K^T$: (32,8,64,128) *transpose last dim
- so $((128,64)@(64,128)) = (128,128)$ *if u confused about the result, open again ur linalg book wkwk ⬇️
    - if 2 matrix with ordo (AxB) and (CxD) to be multiplied, it requires B=C! so the result is (row the 1st matrix) x (col the 2nd matrix) 
- full tensor become (128,128)
- final --> (32,8,128,128)

- softmax($QK^T$) -> it not change the shape
- multiply with $V$
    - (32,8,128,128)@(32,8,128,64) => (128,128)@(128,64) => (32,8,128,64)
- merge head back -> (transpose(1,2)&reshape) -> (32,128,8,64) -> (32,128,512)
- output projection -> `nn.Linear(512,512)` -> shape still (32,128,512) 
- attention block final -> (32,128,512) back in original form 😅

<font style='color: red'> <b >Mini Architecture

> Input: token ids => Output: contextual representation

In [ ]:
class MiniTransformers(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(30000,512)

        self.q_proj = nn.Linear(512,512)
        self.k_proj = nn.Linear(512,512)
        self.v_proj = nn.Linear(512,512)

        self.ffn1 = nn.Linear(512,2048)
        self.ffn2 = nn.Linear(2048,512)

        self.act = nn.GELU()

### Exercise

```
Token IDs
    ↓
Embedding
    ↓
QKV projection
    ↓
Attention-ish flow
    ↓
FFN
    ↓
Output
```

In [5]:
class SelfAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.q_proj = nn.Linear(16,16)
        self.k_proj = nn.Linear(16,16)
        self.v_proj = nn.Linear(16,16)

    def forward(self,x):
        print("\nINPUT")
        print(x.shape)

        # QKV PROJECTION
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        print("\nQ SHAPE")
        print(Q.shape)

        print("\nK SHAPE")
        print(K.shape)

        print("\nV SHAPE")
        print(V.shape)

        # ATTENTION SCORES
        scores = Q@K.transpose(-2,-1)
        
        print("\nSCORES SHAPE")
        print(scores.shape)

        # SCALE
        d_k = K.shape[-1]
        scores = scores / (d_k ** 0.5)

        # SOFTMAX
        weights = F.softmax(scores,dim=-1)

        print("\nATTENTION WEIGHTS")
        print(weights.shape)

        # WEIGHTS SUM
        out = weights @ V

        print("\nFINAL OUTPUT")
        print(out.shape)

        return out

In [3]:
x = torch.randn(2,5,16)

In [7]:
model = SelfAttention()
out = model(x)


INPUT
torch.Size([2, 5, 16])

Q SHAPE
torch.Size([2, 5, 16])

K SHAPE
torch.Size([2, 5, 16])

V SHAPE
torch.Size([2, 5, 16])

SCORES SHAPE
torch.Size([2, 5, 5])

ATTENTION WEIGHTS
torch.Size([2, 5, 5])

FINAL OUTPUT
torch.Size([2, 5, 16])


## CNN - Conv2D

- Input image: (B,C,H,W) -> (32,3,224,224)
- Layer will be:
    ```
    nn.Conv2d(
    in_channels=3,
    out_channels=64,
    kernel_size=3)
- Conv make the size channel changed -> spatial layer will be smaller -> (32,3,222,222)
- 222 where does that come?
    - by default stride=1 and padding=0
    - formula: $H_{out} = \frac{H+2P-C}{s}+1$ = $\frac{224-3}{1}+1 = 222$
- Naa we've layer previously with (3,64,3) -> (in_channels, out_channels, kernel_size)
- conv layer simply change image into collection of feature map
    - 3 channel RGB -> 64 feature maps
- so the shape input:(32,3,222,222) -> layer:(3,64,3) -> out:(32,64,H',W')
    - h and w changed bcuz convultion
    - kernel always "eat" edge side(kernel size=3) -> filter 3x3
- padding add addtional border so (32,64,224,224)
- pooling make channel remain and smaller spatial

<b> CNN overall like this <br>
```
    Input
(32,3,224,224)
    ↓ Conv
(32,64,224,224)
    ↓ Pool
(32,64,112,112)
    ↓ Conv
(32,128,112,112)
    ↓ Pool
(32,128,56,56)
    ↓ Flatten
(32,128,7,7) *due to entered linear layer must be flatten
    ↓ x = torch.flatten(x,start_dim=1) (32,128*7*7)
(32,6272)
nn.Linear(6272,1000) *1000 its out class (ImageNet)
```

<font style='color: red'> <b >Mini Architecture

> conv: explore and learn about feature => pool: compress spatial => linear: classification reasoning

In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        # (B,C,H,W) initial => (32,3,224,224)
        super().__init__()
        self.conv1 = nn.Conv2d(3,32,3,padding=1) #in,out,kernel
        self.pool1 = nn.MaxPool2d(2)
        # B,C,H',W' => (32,32,112,112) 32 cuz learned 32 feature maps 112 affected by pool 224/2
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.pool2 = nn.MaxPool2d(2)
        # B,C,H',W' => (32,64,56,56)
        self.conv3 = nn.Conv2d(64,128,3,padding=1)
        self.flatten = nn.Flatten()
        # B,C,H',W' => (32,128,28,28)
        self.fc = nn.Linear(128*28*28,10) 

    def forward(self,x):
        x = self.conv1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

## Encoder vs Decoder

### Encoder 🕹️

In [2]:
# dummy data
VOCAB = {
    '[PAD]': 0, '[CLS]': 1, '[SEP]': 2, '[MASK]': 3,
    'The': 4, 'the': 5, 'cat': 6, 'sat': 7, 'on': 8,
    'mat': 9, 'dog': 10, 'ran': 11, 'fast': 12,
    'a': 13, 'big': 14, 'small': 15,
}

ID2WORD = {v:k for k,v in VOCAB.items()}
VOCAB_SIZE = len(VOCAB)

# dummy sentences - 2 samples in one batch
# BERT FORMAT: [CLS] token1 token2 token3 ... [SEP]
# we mask certain position and model predict real token

In [3]:
# let say, real token: "The cat sat on the mat" and "The dog ran fast"
# after masking phase: "The dog [MASK] sat on the mat"  and "The dog [MASK] fast"
 
SENTENCES_MASKED = [
    # [CLS], The, [MASK], sat,  on,   the,  mat,  [SEP]
    [1,      4,   3,      7,    8,    5,    9,    2],
    # [CLS], The, dog,   [MASK], fast, [PAD], [PAD], [SEP]
    [1,      4,   10,    3,      12,   0,     0,     2],
]

SENTENCES_ORIGINAL = [
    [1, 4, 6,  7, 8, 5, 9,  2],   # label: 2nd position = cat (id=6)
    [1, 4, 10, 7, 12, 0, 0, 2],   # label: 3rd position = ran (id=11)
]

# Label: integer id from real token on position that masked
# -100 = ignored CrossEntropyLoss (non-mask position)
MASK_LABELS = [
    [-100, -100, 6,   -100, -100, -100, -100, -100],  # => "cat"
    [-100, -100, -100, 7,   -100, -100, -100, -100],  # => "ran"
]

In [4]:
input_ids  = torch.tensor(SENTENCES_MASKED)  # (B=2, T=8)
labels     = torch.tensor(MASK_LABELS)        # (B=2, T=8)
attn_mask  = (input_ids != VOCAB['[PAD]']).long()  # (B=2, T=8) — 0 di posisi PAD
 
print("=" * 60)
print("ENCODER (BERT-style MLM) FROM SCRATCH")
print("=" * 60)
print(f"\n[DATA]")
print(f"  input_ids   shape : {input_ids.shape}   dtype: {input_ids.dtype}")
print(f"  labels      shape : {labels.shape}   dtype: {labels.dtype}")
print(f"  attn_mask   shape : {attn_mask.shape}   dtype: {attn_mask.dtype}")
print(f"\n  Sentence 0 tokens : {[ID2WORD[i] for i in SENTENCES_MASKED[0]]}")
print(f"  Sentence 1 tokens : {[ID2WORD[i] for i in SENTENCES_MASKED[1]]}")

ENCODER (BERT-style MLM) FROM SCRATCH

[DATA]
  input_ids   shape : torch.Size([2, 8])   dtype: torch.int64
  labels      shape : torch.Size([2, 8])   dtype: torch.int64
  attn_mask   shape : torch.Size([2, 8])   dtype: torch.int64

  Sentence 0 tokens : ['[CLS]', 'The', '[MASK]', 'sat', 'on', 'the', 'mat', '[SEP]']
  Sentence 1 tokens : ['[CLS]', 'The', 'dog', '[MASK]', 'fast', '[PAD]', '[PAD]', '[SEP]']


#### Embedding

In [5]:
class EmbeddingLayer(nn.Module):
    '''
    Join embedding token + positional embedding
    BERT using pos embedding approach (not sinusiodal) so that both of them added up not concat (shape remain B,T,d_model)
    '''

    def __init__(self,vocab_size,d_model,max_len,dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size,d_model,padding_idx=0)
        # padding idx=0 -> vector [PAD] always zero
        self.pos_emb = nn.Embedding(max_len,d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self,token_ids):
        # token_ids = (B,T)
        B,T = token_ids.shape

        # lookup token embeddings
        tok = self.token_emb(token_ids) #(B,T,d_model)

        # make position [0, 1, 2, ..., T-1] then lookup
        positions = torch.arange(T, device=token_ids.device)   # (T,)
        pos = self.pos_emb(positions)          # (T, d_model)
        # pos broadcast to (B, T, d_model) while added up
        x = self.dropout(self.norm(tok + pos)) # (B, T, d_model)
        return x

#### Multi Head Attention

In [7]:
class MultiHeadSelfAttention(nn.Module):
    '''
    Bidirectional self att - there is no casual mask

    shape flow
        input x : (B,T,d_model)
        Q,K,V proj : (B,T,d_model) -> split heads: (B,H,T,d_head) 
            -> Q@K^T = (B,H,T,d_head)*(B,H,d_head,T) = (B,H,T,T)
        scores : (B,H,T,T)
        attn_weights : (B,H,T,T) remains even apply softmax
        context : (B,H,T,d_head)
        output : (B,T,d_model) after merge heads + out projection
    '''

    def __init__(self,d_model,n_heads,dropout=0.1):
        super().__init__()
        assert d_model%n_heads == 0, "d_model must be divisible by n_heads"

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads #dimension per head

        # Q,K,V projection - can be combine to one linear(d,3d), but here we seperated so can be more readible
        self.W_q = nn.Linear(d_model,d_model,bias=False)
        self.W_k = nn.Linear(d_model,d_model,bias=False)
        self.W_v = nn.Linear(d_model,d_model,bias=False)
        self.W_o = nn.Linear(d_model,d_model,bias=False) #out projection

        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.head_dim) # 1/sqrt(d_k) scaling factor

    def forward(self, x, padding_mask=None):
        # x            : (B, T, d_model)
        # padding_mask : (B, T) — True in position that must be ignore (PAD tokens)
 
        B, T, _ = x.shape
        H        = self.n_heads
        hd       = self.head_dim
 
        # ── Step 1: Linear projections ─────────────────────────────────────
        Q = self.W_q(x)   # (B, T, d_model)
        K = self.W_k(x)   # (B, T, d_model)
        V = self.W_v(x)   # (B, T, d_model)
 
        # ── Step 2: Split to multiple heads ────────────────────────────────
        # (B, T, d_model) → (B, T, H, head_dim) → (B, H, T, head_dim)
        Q = Q.view(B, T, H, hd).transpose(1, 2)   # (B, H, T, head_dim)
        K = K.view(B, T, H, hd).transpose(1, 2)   # (B, H, T, head_dim)
        V = V.view(B, T, H, hd).transpose(1, 2)   # (B, H, T, head_dim)
 
        # ── Step 3: Scaled dot-product attention ───────────────────────────
        # (B, H, T, head_dim) @ (B, H, head_dim, T) = (B, H, T, T)
        scores = (Q @ K.transpose(-2, -1)) / self.scale   # (B, H, T, T)
 
        # ── Step 4: Apply padding mask ─────────────────────────────────────
        if padding_mask is not None:
            # padding_mask: (B, T) → expand ke (B, 1, 1, T) untuk broadcast
            # posisi PAD di KEY dimension → set ke -inf supaya softmax → ~0
            pad_mask = padding_mask.unsqueeze(1).unsqueeze(2)   # (B, 1, 1, T)
            scores = scores.masked_fill(pad_mask == 0, float('-inf'))
 
        # ── Step 5: Softmax → attention weights ────────────────────────────
        attn_weights = F.softmax(scores, dim=-1)   # (B, H, T, T)
        attn_weights = self.dropout(attn_weights)
 
        # ── Step 6: Weighted sum of values ─────────────────────────────────
        # (B, H, T, T) @ (B, H, T, head_dim) = (B, H, T, head_dim)
        context = attn_weights @ V   # (B, H, T, head_dim)
 
        # ── Step 7: Merge heads ─────────────────────────────────────────────
        # (B, H, T, head_dim) → transpose → (B, T, H, head_dim)
        # → contiguous() karena setelah transpose data tidak contiguous
        # → view → (B, T, d_model)
        context = context.transpose(1, 2).contiguous()   # (B, T, H, head_dim)
        context = context.view(B, T, self.d_model)        # (B, T, d_model)
 
        # ── Step 8: Output projection ───────────────────────────────────────
        out = self.W_o(context)   # (B, T, d_model)
 
        return out, attn_weights  # return weights untuk inspeksi

#### FFN

In [8]:
class FeedForward(nn.Module):
    '''
    2 layer MLP with GELU act function
    Expand then contract : d_model -> d_ff -> d_model

    Shape flow:
        input: (B,T,d_model)
        linear1: (B,T,d_ff) <- start expand
        GELU: (B,T,d_ff) <- non linearity
        linear2: (B,T,d_model)
    '''

    def __init__(self,d_model,d_ff,dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model,d_ff)
        self.linear2 = nn.Linear(d_model,d_ff)
        self.dropout = nn.Dropout(dropout)
        self.act = nn.GELU()

    def forward(self,x):
        # x : (B,T,d_model)
        x = self.linear1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

#### Encoder Block DANGEERRRR

In [9]:
class EncoderBlock(nn.Module):
    """
    Satu Transformer encoder block:
        x → LayerNorm → MHA → +residual
          → LayerNorm → FFN → +residual
 
    Pre-norm style (normalize input, bukan output).
    Input shape = Output shape = (B, T, d_model).
    """
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = FeedForward(d_model, d_ff, dropout)
        self.drop  = nn.Dropout(dropout)
 
    def forward(self, x, padding_mask=None):
        # x : (B, T, d_model)
 
        # ── Attention sublayer ─────────────────────────────────────────────
        residual = x                              # (B, T, d_model) simpan untuk skip
        x_norm   = self.norm1(x)                  # (B, T, d_model)
        attn_out, attn_w = self.attn(x_norm, padding_mask)
        x = residual + self.drop(attn_out)        # (B, T, d_model) + skip
 
        # ── FFN sublayer ───────────────────────────────────────────────────
        residual = x                              # (B, T, d_model)
        x_norm   = self.norm2(x)                  # (B, T, d_model)
        ffn_out  = self.ffn(x_norm)               # (B, T, d_model)
        x = residual + self.drop(ffn_out)         # (B, T, d_model) + skip
 
        return x, attn_w   # (B, T, d_model), (B, H, T, T)


In [10]:
class BERTEncoder(nn.Module):
    """
    Full encoder model = Embedding + N × EncoderBlock + MLM Head.
 
    MLM Head:
        hidden (B, T, d_model) → Linear(d_model, vocab_size) → logits (B, T, vocab_size)
        Loss computed only at [MASK] positions (label = -100 elsewhere).
    """
    def __init__(
        self,
        vocab_size : int,
        d_model    : int  = 64,
        n_heads    : int  = 4,
        d_ff       : int  = 128,
        n_layers   : int  = 2,
        max_len    : int  = 64,
        dropout    : float = 0.1,
    ):
        super().__init__()
        self.embedding = EmbeddingLayer(vocab_size, d_model, max_len, dropout)
 
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
 
        self.norm = nn.LayerNorm(d_model)   # final normalization
 
        # MLM head
        self.mlm_head = nn.Linear(d_model, vocab_size)
 
    def forward(self, input_ids, attention_mask=None):
        """
        input_ids      : (B, T) int64
        attention_mask : (B, T) int64 — 1 = real token, 0 = padding
 
        Returns:
            logits : (B, T, vocab_size)
            loss   : scalar (kalau labels diberikan)
        """
        # ── Step 1: Embedding ───────────────────────────────────────────────
        x = self.embedding(input_ids)   # (B, T, d_model)
 
        # ── Step 2: N Encoder Blocks ────────────────────────────────────────
        all_attn_weights = []
        for layer in self.layers:
            x, attn_w = layer(x, attention_mask)   # (B, T, d_model)
            all_attn_weights.append(attn_w)
 
        # ── Step 3: Final LayerNorm ─────────────────────────────────────────
        x = self.norm(x)   # (B, T, d_model)
 
        # ── Step 4: MLM Head ────────────────────────────────────────────────
        logits = self.mlm_head(x)   # (B, T, vocab_size)
 
        return logits, all_attn_weights
 

In [11]:
def train_encoder():
    print("\n" + "=" * 60)
    print("ENCODER TRAINING LOOP")
    print("=" * 60)
 
    # Config
    D_MODEL = 64
    N_HEADS = 4
    D_FF    = 128
    N_LAYERS = 2
 
    model = BERTEncoder(
        vocab_size = VOCAB_SIZE,
        d_model    = D_MODEL,
        n_heads    = N_HEADS,
        d_ff       = D_FF,
        n_layers   = N_LAYERS,
    )
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\nModel params: {total_params:,}")
 
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    # ignore_index=-100 → loss tidak dihitung di posisi non-mask
 
    # ── Trace shapes satu forward pass ─────────────────────────────────────
    print(f"\n[SHAPE TRACE — satu forward pass]")
    print(f"  input_ids         : {input_ids.shape}")
 
    model.eval()
    with torch.no_grad():
        # Manual trace per layer
        emb_out = model.embedding(input_ids)
        print(f"  after embedding   : {emb_out.shape}   ← (B, T, d_model={D_MODEL})")
 
        x = emb_out
        for i, layer in enumerate(model.layers):
            x, attn_w = layer(x, attn_mask)
            print(f"  after layer {i+1}     : {x.shape}   attn_w: {attn_w.shape}")
 
        x = model.norm(x)
        print(f"  after final norm  : {x.shape}")
 
        logits = model.mlm_head(x)
        print(f"  logits            : {logits.shape}   ← (B, T, vocab={VOCAB_SIZE})")
 
        # Prediction di posisi MASK
        pred_s0 = logits[0, 2, :].argmax().item()   # kalimat 0, posisi 2
        pred_s1 = logits[1, 3, :].argmax().item()   # kalimat 1, posisi 3
        print(f"\n  [BEFORE TRAINING — prediction at [MASK] positions]")
        print(f"  Sentence 0 pos 2: predicted='{ID2WORD[pred_s0]}' | expected='cat'")
        print(f"  Sentence 1 pos 3: predicted='{ID2WORD[pred_s1]}' | expected='ran'")
 
    # ── Training loop ───────────────────────────────────────────────────────
    model.train()
    print(f"\n[TRAINING — 200 steps]")
    for step in range(200):
        logits, _ = model(input_ids, attn_mask)
        # logits : (B, T, vocab_size)
        # labels : (B, T) — -100 di posisi non-mask
 
        # CrossEntropyLoss expects:
        # input : (N, C) → kita perlu reshape (B*T, vocab_size)
        # target: (N,)   → kita perlu reshape (B*T,)
        loss = criterion(
            logits.view(-1, VOCAB_SIZE),   # (B*T, vocab_size) = (16, 16)
            labels.view(-1)                # (B*T,) = (16,)
        )
 
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
 
        if step % 40 == 0:
            # Cek prediksi di [MASK] positions
            with torch.no_grad():
                pred0 = logits[0, 2, :].argmax().item()
                pred1 = logits[1, 3, :].argmax().item()
            print(f"  step {step:3d} | loss: {loss.item():.4f} | "
                  f"pred0='{ID2WORD[pred0]}' pred1='{ID2WORD[pred1]}'")
 
    # ── Final evaluation ────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        logits, attn_weights = model(input_ids, attn_mask)
 
        pred0 = logits[0, 2, :].argmax().item()
        pred1 = logits[1, 3, :].argmax().item()
 
        print(f"\n[FINAL RESULT]")
        print(f"  Sentence 0: 'The [MASK] sat on the mat'")
        print(f"    → predicted: '{ID2WORD[pred0]}'  (expected: 'cat')  "
              f"{'✓' if pred0 == VOCAB['cat'] else '✗'}")
        print(f"  Sentence 1: 'The dog [MASK] fast'")
        print(f"    → predicted: '{ID2WORD[pred1]}'  (expected: 'ran')  "
              f"{'✓' if pred1 == VOCAB['ran'] else '✗'}")
 
        # Attention pattern di layer terakhir, head pertama
        last_attn = attn_weights[-1][0, 0, :, :]   # (T, T) head 0, sentence 0
        print(f"\n[ATTENTION PATTERN — last layer, head 0, sentence 0]")
        print(f"  shape: {last_attn.shape}  (T×T attention matrix)")
        print(f"  rows = query tokens, cols = key tokens")
        header = "  " + " ".join(f"{ID2WORD[i]:>6}" for i in SENTENCES_MASKED[0])
        print(header)
        for qi, qtok in enumerate(SENTENCES_MASKED[0]):
            row = f"  {ID2WORD[qtok]:>6} "
            row += " ".join(f"{last_attn[qi, ki].item():6.3f}"
                            for ki in range(len(SENTENCES_MASKED[0])))
            print(row)

In [12]:
train_encoder()


ENCODER TRAINING LOOP

Model params: 72,976

[SHAPE TRACE — satu forward pass]
  input_ids         : torch.Size([2, 8])
  after embedding   : torch.Size([2, 8, 64])   ← (B, T, d_model=64)


RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x128 and 64x128)

### Decoder ⚾︎